# Silent Signal — ASL Citizen trên Google Colab

Notebook **chuẩn bị và kiểm tra dữ liệu**, chưa trích xuất pose hoặc huấn luyện. Source được lấy từ nhánh `dev`; mọi manifest, label map, split, cache và báo cáo được ghi trực tiếp vào Google Drive. Dataset tải về ổ tạm Colab, không cần Zenodo token.

Giữ nguyên train/val/test chính thức: **40.154 / 10.304 / 32.941 video**, **35 / 6 / 11 signer**, tổng 83.399 video và 2.731 nhãn Gloss. `val` được biểu diễn là `validation` trong manifest chung. Không chia ngẫu nhiên lại và không dùng ASL-LEX Code làm lớp.

Nguồn: [Microsoft Research](https://www.microsoft.com/en-us/research/project/asl-citizen/), [datasheet](https://www.microsoft.com/en-us/research/project/asl-citizen/datasheet/), [điều khoản dataset](https://www.microsoft.com/en-us/research/project/asl-citizen/dataset-license/). Đọc điều khoản trước khi tải/sử dụng; lưu ý phạm vi nghiên cứu phi thương mại. ASL Citizen chứa ngôn ngữ ký hiệu Mỹ.

**Chạy từng cell lần đầu.** Sau khi phần ASL Citizen được push/merge lên `dev`, mở notebook này từ GitHub trong Colab. CPU đủ cho giai đoạn này. Nếu Colab không đủ chỗ giữ cả ZIP và dataset, dùng `STREAM_EXTRACT=True` để giải nén trực tiếp từ Microsoft vào `/content`; các cờ thao tác mặc định tắt để bạn kiểm tra cấu hình trước.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'dev'  # @param {type:'string'}
PROJECT_ROOT = Path('/content/silent-signal')

DATASET_ROOT = Path('/content/ASL_Citizen')
ARCHIVE_PATH = Path('/content/asl-citizen-release/ASL_Citizen.zip')
RESULT_ROOT = Path('/content/drive/MyDrive/silent-signal-results/asl_citizen')

STREAM_EXTRACT = False  # @param {type:'boolean'}
DOWNLOAD = False  # @param {type:'boolean'}
EXTRACT = False  # @param {type:'boolean'}
RUN_PROBE = False  # @param {type:'boolean'}
RUN_FULL_DECODE = False  # @param {type:'boolean'}
MEDIA_WORKERS = 2  # @param {type:'integer'}
MEDIA_BATCH_SIZE = 250  # @param {type:'integer'}
# keep: tiếp tục cache; restart: tạo thư mục cache mới, giữ lại cache cũ.
CACHE_MODE = 'keep'  # @param ['keep', 'restart']
RETRY_MEDIA_ERRORS = False  # @param {type:'boolean'}
# Tối đa video MỚI được decode mỗi lần chạy cell; 0 nghĩa là toàn bộ còn lại.
DECODE_LIMIT = 0  # @param {type:'integer'}

if STREAM_EXTRACT and (DOWNLOAD or EXTRACT):
    raise ValueError('STREAM_EXTRACT không được bật cùng DOWNLOAD hoặc EXTRACT.')
if MEDIA_WORKERS < 1 or MEDIA_BATCH_SIZE < 1 or DECODE_LIMIT < 0:
    raise ValueError('Workers/batch phải dương; DECODE_LIMIT không được âm.')
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_ROOT)
print('Kết quả lưu trên Drive:', RESULT_ROOT)


## Clone hoặc cập nhật đúng nhánh dev

Cell kiểm tra repository, nhánh và thay đổi local trước khi cập nhật bằng fast-forward. Nếu báo nhánh chưa có hỗ trợ ASL Citizen, hãy push/merge code mới lên GitHub rồi chạy lại. Với repository private, cần xác thực GitHub trong môi trường Colab theo cách của bạn; không đưa token vào URL hoặc notebook.


In [ ]:
import json
import shutil
import subprocess
import sys

def git_output(*arguments):
    return subprocess.check_output(
        ['git', '-C', str(PROJECT_ROOT), *arguments], text=True
    ).strip()

if not PROJECT_ROOT.exists():
    subprocess.run(
        ['git', 'clone', '--branch', PROJECT_GIT_REF, '--depth', '1',
         PROJECT_GIT_URL, str(PROJECT_ROOT)], check=True
    )
else:
    if not (PROJECT_ROOT / '.git').is_dir():
        raise RuntimeError('PROJECT_ROOT đã tồn tại nhưng không phải Git repo. Chọn đường dẫn khác.')
    if git_output('remote', 'get-url', 'origin') != PROJECT_GIT_URL:
        raise RuntimeError('Repo local có origin khác cấu hình. Kiểm tra PROJECT_ROOT.')
    if git_output('branch', '--show-current') != PROJECT_GIT_REF:
        raise RuntimeError('Repo local đang ở nhánh khác. Chọn PROJECT_ROOT mới hoặc xử lý Git trước.')
    if git_output('status', '--porcelain'):
        raise RuntimeError('Repo local có thay đổi chưa commit. Hãy lưu thay đổi trước khi cập nhật.')
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', PROJECT_GIT_REF],
        check=True
    )

SOURCE_CONFIG = PROJECT_ROOT / 'configs/dataset/asl_citizen.yaml'
if not SOURCE_CONFIG.is_file() or not (
    PROJECT_ROOT / 'src/silent_signal/data/asl_download.py'
).is_file():
    raise RuntimeError(
        'Nhánh đã tải chưa có code ASL Citizen. Push/merge thay đổi lên dev, rồi chạy lại cell.'
    )
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(PROJECT_ROOT)],
    check=True
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if shutil.which('ffprobe') is None or shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
PROJECT_COMMIT = git_output('rev-parse', 'HEAD')
print('Source commit:', PROJECT_COMMIT)
print('ffprobe:', shutil.which('ffprobe'))


## Chọn tải ZIP hoặc giải nén trực tiếp

**Khuyến nghị cho Colab dung lượng hạn chế:** bật `STREAM_EXTRACT=True`, giữ `DOWNLOAD=False` và `EXTRACT=False`. ZIP được đọc theo từng vùng HTTP và giải nén thẳng vào `DATASET_ROOT`, nên chỉ cần khoảng **48,20 GiB** gồm 2 GiB dự phòng. Không ngắt/xóa runtime trong khi chạy vì `/content` là ổ tạm.

Chỉ dùng luồng cũ khi có ổ lưu trữ phù hợp: `DOWNLOAD=True` tải ZIP bản 1.0 khoảng **42,77 GiB** (45.924.134.223 byte), sau đó `EXTRACT=True` giải nén khoảng **46,20 GiB**. Nếu giữ cả hai trên cùng ổ, cần khoảng **91 GiB**.

Tải dở được lưu thành `.part`, kèm metadata để kiểm tra file trên server có thay đổi không. Khi mạng ngắt, chạy lại cell để tiếp tục. Cell không tự xóa ZIP. Dữ liệu trong `/content` mất khi Colab xóa runtime; cache/báo cáo trên Drive vẫn còn. Có thể đổi ARCHIVE_PATH hoặc DATASET_ROOT sang ổ lưu trữ đủ dung lượng.


In [ ]:
from silent_signal.data.asl_download import download_archive, inspect_archive

ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Ổ chứa ZIP còn {shutil.disk_usage(ARCHIVE_PATH.parent).free / 1024**3:.2f} GiB')
if DOWNLOAD:
    # Ước lượng trước khi tải; cell giải nén vẫn kiểm tra dung lượng chính xác.
    v1_extracted_bytes = 49_604_368_459
    existing_bytes = sum(p.stat().st_size for p in DATASET_ROOT.rglob('*') if p.is_file())
    extraction_remaining = max(0, v1_extracted_bytes - existing_bytes)
    reserve_bytes = 2 * 1024**3
    if EXTRACT:
        if DATASET_ROOT.stat().st_dev == ARCHIVE_PATH.parent.stat().st_dev:
            reserve_bytes += extraction_remaining
        elif shutil.disk_usage(DATASET_ROOT).free < extraction_remaining + reserve_bytes:
            raise RuntimeError('Ổ DATASET_ROOT không đủ chỗ giải nén. Đổi đường dẫn lưu trữ.')
    download_archive(ARCHIVE_PATH, reserve_bytes=reserve_bytes)
elif STREAM_EXTRACT:
    print('STREAM_EXTRACT=True: bỏ qua tải ZIP; cell tiếp theo sẽ giải nén trực tiếp.')
else:
    print('DOWNLOAD=False: bỏ qua tải. Bật True nếu chưa có ZIP hoặc dataset.')

if ARCHIVE_PATH.is_file():
    archive_info = inspect_archive(ARCHIVE_PATH)
    print('Thư mục gốc trong ZIP:', archive_info.prefix or '(không có thư mục bao ngoài)')
    print('Số file cần giải nén:', archive_info.file_count)
    print(f'Dung lượng nội dung: {archive_info.uncompressed_bytes / 1024**3:.2f} GiB')
elif STREAM_EXTRACT:
    print('Không cần ZIP local khi STREAM_EXTRACT=True.')
else:
    print('Chưa có ZIP. Có thể tiếp tục nếu DATASET_ROOT đã chứa dữ liệu giải nén.')


## Giải nén

Với `STREAM_EXTRACT=True`, cell mở ZIP từ xa bằng HTTP Range, ghim ETag/Last-Modified của release và giải nén theo thứ tự archive để giảm request. Với luồng ZIP local, đổi `EXTRACT=True` sau khi tải xong. Cả hai luồng đều kiểm tra đường dẫn, dung lượng và CRC từng file; chạy lại sau gián đoạn sẽ chỉ bỏ qua file có size và CRC khớp.


In [ ]:
from silent_signal.data.asl_download import extract_archive, extract_remote_archive

if STREAM_EXTRACT:
    extract_remote_archive(DATASET_ROOT)
elif EXTRACT:
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError('Chưa có ZIP hoàn chỉnh. Hoàn tất cell DOWNLOAD trước.')
    extract_archive(ARCHIVE_PATH, DATASET_ROOT)
else:
    print('EXTRACT=False: dùng dataset đã có hoặc bật True để giải nén.')

required_paths = ['videos', 'splits/train.csv', 'splits/val.csv', 'splits/test.csv']
missing = [name for name in required_paths if not (DATASET_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'DATASET_ROOT thiếu {missing}. Kiểm tra cell tải/giải nén.')
if not (DATASET_ROOT / 'videos').is_dir():
    raise NotADirectoryError(DATASET_ROOT / 'videos')
usage_path = DATASET_ROOT / 'use.txt'
if usage_path.is_file():
    print('Điều khoản đi kèm dataset:', usage_path)
    print('Đọc tại: https://www.microsoft.com/en-us/research/project/asl-citizen/dataset-license/')
print('Dataset root:', DATASET_ROOT)


## Tạo cấu hình đầu ra trên Drive, manifest và split chính thức

Cell tạo YAML riêng trên Drive **trước khi chạy CLI**. CSV, Parquet, label map và báo cáo không phụ thuộc vào việc chép file cuối phiên. Pipeline nhập nguyên membership từ `splits/train.csv`, `val.csv`, `test.csv`, kiểm tra số lượng, nhãn và signer không giao nhau.

Lỗi metadata hoặc video thiếu/rỗng sẽ làm lệnh dừng; xem báo cáo trong `reports/`. Không tự loại mẫu hoặc chia lại để làm benchmark hợp lệ. Kiểm tra metadata mới chỉ xác nhận cấu trúc/file, chưa bảo đảm video giải mã được.


In [ ]:
from datetime import datetime, timezone
import yaml

def write_json_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)

config_payload = yaml.safe_load(SOURCE_CONFIG.read_text(encoding='utf-8'))
if config_payload['dataset'].get('adapter') != 'asl_citizen':
    raise RuntimeError('Source config không phải adapter asl_citizen.')
if config_payload['split'].get('strategy') != 'official':
    raise RuntimeError('ASL Citizen cần split.strategy=official.')
config_payload['dataset']['root'] = str(DATASET_ROOT)
config_payload['outputs'] = {
    'manifest_csv': str(RESULT_ROOT / 'manifests/asl_citizen.csv'),
    'manifest_parquet': str(RESULT_ROOT / 'manifests/asl_citizen.parquet'),
    'labels': str(RESULT_ROOT / 'labels/asl_citizen_labels.json'),
    'split': str(RESULT_ROOT / 'splits/asl_citizen_official.json'),
    'report': str(RESULT_ROOT / 'reports/metadata_validation_report.json'),
    'invalid_records': str(RESULT_ROOT / 'reports/metadata_invalid_records.csv'),
}
CONFIG_PATH = RESULT_ROOT / 'configs/asl_citizen.colab.yaml'
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config_payload, allow_unicode=True), encoding='utf-8')

STATUS_PATH = RESULT_ROOT / 'reports/preparation_status.json'
status = {
    'dataset': 'asl_citizen',
    'source_commit': PROJECT_COMMIT,
    'dataset_root': str(DATASET_ROOT),
    'started_at': datetime.now(timezone.utc).isoformat(),
    'metadata': {'state': 'running'},
    'probe': {'state': 'not_checked_this_run'},
    'decode': {'state': 'not_checked_this_run'},
}
write_json_atomic(STATUS_PATH, status)
completed = subprocess.run(
    [sys.executable, '-m', 'silent_signal.cli.prepare', 'all',
     '--config', str(CONFIG_PATH), '--root', str(DATASET_ROOT), '--level', 'metadata'],
    cwd=PROJECT_ROOT, text=True, capture_output=True
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
status['metadata'] = {'state': 'passed' if completed.returncode == 0 else 'failed'}
write_json_atomic(STATUS_PATH, status)
if completed.returncode:
    raise RuntimeError(f'Metadata chưa đạt. Xem báo cáo tại {RESULT_ROOT / "reports"}.')
print('Manifest và split chính thức đã lưu:', RESULT_ROOT)


In [ ]:
from silent_signal.configuration import load_dataset_config
from silent_signal.data.manifest import read_manifest, manifest_summary

dataset_config = load_dataset_config(CONFIG_PATH, root_override=DATASET_ROOT)
records = read_manifest(RESULT_ROOT / 'manifests/asl_citizen.parquet')
print(json.dumps(manifest_summary(records), ensure_ascii=False, indent=2))
print('Nhãn và split không thay đổi giữa các bước metadata/probe/decode.')


## Bộ nhớ kiểm tra video có thể tiếp tục

Các cell sau dùng lại `probe_video`, `decode_video` và bộ validation chung. Kết quả được ghi theo batch trên Drive; mỗi entry gắn với dataset, đường dẫn, nhãn/split và size/mtime của video. File thay đổi sẽ được kiểm tra lại. Cache giúp tiếp tục công việc, không lưu video; runtime mất video thì cần tải/giải nén lại.

`CACHE_MODE='keep'` dùng cache trước; `'restart'` tạo thư mục cache mới và giữ thư mục cũ. `RETRY_MEDIA_ERRORS=True` cho phép kiểm tra lại các lỗi đã cache sau khi bạn xử lý nguyên nhân. Khi chạy lại cell này ở chế độ restart, một lượt cache mới được tạo.


In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict

from silent_signal.data.validation import (
    MediaInfo, MediaProbeError, decode_video, probe_video,
    validate_manifest, write_validation_report,
)
from silent_signal.data.manifest import write_manifest

cache_base = RESULT_ROOT / 'cache'
cache_pointer = cache_base / 'active.json'
if CACHE_MODE not in {'keep', 'restart'}:
    raise ValueError("CACHE_MODE phải là 'keep' hoặc 'restart'.")
if CACHE_MODE == 'keep' and cache_pointer.is_file():
    cache_name = json.loads(cache_pointer.read_text(encoding='utf-8'))['directory']
    if Path(cache_name).name != cache_name or cache_name in {'.', '..'}:
        raise ValueError('Đường dẫn cache không hợp lệ.')
else:
    cache_name = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    write_json_atomic(cache_pointer, {'directory': cache_name})
CACHE_ROOT = cache_base / cache_name
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print('Cache đang dùng:', CACHE_ROOT)

def video_signature(record):
    path = DATASET_ROOT / record.video_path
    info = path.stat() if path.is_file() else None
    return {
        'schema_version': 1, 'dataset': 'asl_citizen',
        'source_commit': PROJECT_COMMIT, 'dataset_root': str(DATASET_ROOT.resolve()),
        'video_path': record.video_path,
        'signer_id': record.signer_id, 'gloss_id': record.gloss_id,
        'split': record.split,
        'size': info.st_size if info else None,
        'mtime_ns': info.st_mtime_ns if info else None,
    }

def cache_is_current(record, item):
    return isinstance(item, dict) and item.get('signature') == video_signature(record)

def load_cache(path):
    cache = {}
    if path.is_file():
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                if not isinstance(item, dict) or 'sample_id' not in item:
                    raise ValueError('Invalid entry')
                cache[item['sample_id']] = item
            except (ValueError, TypeError):
                print(f'Bỏ qua dòng cache không hoàn chỉnh: {line_number}')
    return cache

def inspect_record(record, operation):
    signature = video_signature(record)
    try:
        result = probe_video(DATASET_ROOT / record.video_path) if operation == 'probe' else (
            decode_video(DATASET_ROOT / record.video_path)
        )
        if signature != video_signature(record):
            raise RuntimeError('Video thay đổi trong lúc kiểm tra.')
        return {
            'sample_id': record.sample_id, 'signature': signature, 'status': 'ok',
            'media': asdict(result) if operation == 'probe' else None,
        }
    except Exception as exc:
        return {
            'sample_id': record.sample_id, 'signature': signature, 'status': 'error',
            'error': f'{type(exc).__name__}: {exc}'[:2000],
        }

def current_cache_summary(cache):
    entries = [
        cache[r.sample_id] for r in records
        if cache_is_current(r, cache.get(r.sample_id))
    ]
    return {
        'total': len(records), 'checked': len(entries),
        'errors': sum(item.get('status') != 'ok' for item in entries),
        'remaining': len(records) - len(entries),
    }

def run_media_cache(operation, enabled, limit=0):
    path = CACHE_ROOT / f'{operation}.jsonl'
    cache = load_cache(path)
    current = {r.sample_id: cache[r.sample_id] for r in records
               if cache_is_current(r, cache.get(r.sample_id))}
    pending = [
        r for r in records if r.sample_id not in current
        or (RETRY_MEDIA_ERRORS and current[r.sample_id].get('status') != 'ok')
    ]
    if limit:
        pending = pending[:limit]
    if enabled:
        for start in range(0, len(pending), MEDIA_BATCH_SIZE):
            batch = pending[start:start + MEDIA_BATCH_SIZE]
            with ThreadPoolExecutor(max_workers=MEDIA_WORKERS) as executor:
                futures = [executor.submit(inspect_record, r, operation) for r in batch]
                with path.open('a', encoding='utf-8') as output:
                    # Tách khỏi đuôi JSON dang dở nếu phiên trước bị ngắt giữa một dòng.
                    output.write('\n')
                    for future in as_completed(futures):
                        item = future.result()
                        output.write(json.dumps(item, ensure_ascii=False) + '\n')
                        cache[item['sample_id']] = item
                        current[item['sample_id']] = item
                    output.flush()
            summary = {
                'total': len(records), 'checked': len(current),
                'errors': sum(item.get('status') != 'ok' for item in current.values()),
                'remaining': len(records) - len(current),
            }
            status[operation] = {'state': 'partial', **summary}
            write_json_atomic(STATUS_PATH, status)
            print(operation, summary)
    summary = current_cache_summary(cache)
    state = 'complete_cache' if summary['remaining'] == 0 else (
        'partial' if summary['checked'] else 'not_run'
    )
    status[operation] = {'state': state, **summary}
    write_json_atomic(STATUS_PATH, status)
    print(operation, status[operation])
    return cache

sample_by_path = {str((DATASET_ROOT / r.video_path).resolve()): r for r in records}

def cached_item(path, cache):
    record = sample_by_path[str(path.resolve())]
    item = cache.get(record.sample_id)
    if not cache_is_current(record, item):
        raise MediaProbeError('Cache thiếu hoặc video đã thay đổi.')
    if item.get('status') != 'ok':
        raise MediaProbeError(item.get('error', 'Media check failed'))
    return item

def persist_validation(result, stage):
    write_manifest(result.records, RESULT_ROOT / f'manifests/asl_citizen.{stage}.csv')
    write_manifest(result.records, RESULT_ROOT / f'manifests/asl_citizen.{stage}.parquet')
    write_manifest(
        tuple(r for r in result.records if not r.is_valid),
        RESULT_ROOT / f'reports/{stage}_invalid_records.csv'
    )
    write_validation_report(result, RESULT_ROOT / f'reports/{stage}_validation_report.json')


## ffprobe toàn bộ video

Bật `RUN_PROBE=True` trong cấu hình rồi chạy cell này. Khi cache đủ toàn bộ video, kết quả được đưa qua các quy tắc validation và ghi `asl_citizen.probed.csv/parquet`, báo cáo và trạng thái. FPS/kích thước được đọc từ từng video, không ép theo VSL400. Cache đủ nhưng có lỗi không đồng nghĩa dữ liệu đạt kiểm tra.


In [ ]:
probe_cache = run_media_cache('probe', RUN_PROBE)
if current_cache_summary(probe_cache)['remaining'] == 0:
    def cached_probe(path):
        return MediaInfo(**cached_item(path, probe_cache)['media'])

    probe_result = validate_manifest(
        records, dataset_root=DATASET_ROOT, expected=dataset_config.expected,
        expected_views=tuple(dataset_config.views), level='probe',
        workers=MEDIA_WORKERS, probe_function=cached_probe,
    )
    persist_validation(probe_result, 'probed')
    status['probe']['state'] = 'failed' if probe_result.has_errors else 'passed'
    write_json_atomic(STATUS_PATH, status)
    print('Probe:', status['probe']['state'])
else:
    print('Probe chưa đủ toàn bộ dữ liệu; chưa xuất manifest probed cho phiên này.')


## Full decode tùy chọn, có resume theo video

Chỉ bật `RUN_FULL_DECODE=True` sau khi probe toàn bộ đã **passed**. Mỗi video được ffmpeg giải mã toàn bộ; thao tác tốn thời gian hơn probe. `DECODE_LIMIT=500` kiểm tra tối đa 500 video mới trong mỗi lần chạy cell; `0` xử lý toàn bộ còn lại. Cache vẫn tích lũy khi chạy lại.

Báo cáo `decode_coverage.json` luôn ghi rõ số video đã kiểm tra, còn thiếu và lỗi. Chỉ khi đã kiểm tra mọi video, cell mới tổng hợp validation mức decode và ghi `asl_citizen.decoded.csv/parquet`. Kiểm tra một phần không được ghi nhận là full dataset đã đạt.


In [ ]:
if RUN_FULL_DECODE and status['probe'].get('state') != 'passed':
    raise RuntimeError('Hãy hoàn tất probe toàn bộ và xử lý lỗi trước khi full decode.')
decode_cache = run_media_cache('decode', RUN_FULL_DECODE, DECODE_LIMIT)
coverage = current_cache_summary(decode_cache)
write_json_atomic(RESULT_ROOT / 'reports/decode_coverage.json', coverage)

if coverage['remaining'] == 0 and status['probe'].get('state') == 'passed':
    def cached_decode(path):
        cached_item(path, decode_cache)

    decode_result = validate_manifest(
        records, dataset_root=DATASET_ROOT, expected=dataset_config.expected,
        expected_views=tuple(dataset_config.views), level='decode',
        workers=MEDIA_WORKERS, probe_function=cached_probe, decode_function=cached_decode,
    )
    persist_validation(decode_result, 'decoded')
    status['decode']['state'] = 'failed' if decode_result.has_errors else 'passed'
    write_json_atomic(STATUS_PATH, status)
    print('Full decode:', status['decode']['state'])
else:
    print('Chưa có kết luận full decode cho toàn bộ dataset:', coverage)


## Kết quả và bước tiếp theo

Đọc `reports/preparation_status.json` để xác định mức kiểm tra hiện tại; tên file từ phiên cũ không tự chứng minh rằng dữ liệu hiện tại đã đạt. Chỉ dùng manifest probed/decoded tương ứng khi trạng thái đó là `passed`. Metadata/probe/decode giữ nguyên nhãn và split chính thức.

Các file dưới `labels/`, `splits/`, `manifests/`, `reports/`, `configs/` và `cache/` đều nằm trong RESULT_ROOT trên Drive. Notebook này chưa chạy pose, chưa tạo checkpoint và chưa train; khi dữ liệu đạt, bước tiếp theo là thử trích xuất pose trên một nhóm nhỏ.


In [ ]:
status['updated_at'] = datetime.now(timezone.utc).isoformat()
status['manifest_sha256'] = hashlib.sha256(
    (RESULT_ROOT / 'manifests/asl_citizen.parquet').read_bytes()
).hexdigest()
status['cache_root'] = str(CACHE_ROOT)
write_json_atomic(STATUS_PATH, status)
print(json.dumps(status, ensure_ascii=False, indent=2))
print('Kết quả:', RESULT_ROOT)
